In [1]:
# =========================
# DD / McNichols OLS feature importance
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm

# -------------------------
# 1. Load all firm CSV files
# -------------------------
BASE_DIR = Path(".").resolve()
INPUT_DIR = BASE_DIR / "acc_components_extracted"

files = sorted(INPUT_DIR.glob("*.csv"))
print(f"Found {len(files)} files in {INPUT_DIR}")

dfs = []
for fp in files:
    try:
        df = pd.read_csv(fp)
        df["source_file"] = fp.name
        dfs.append(df)
    except Exception as e:
        print(f"Skipping {fp.name}: {e}")

panel = pd.concat(dfs, ignore_index=True)
print(f"Combined shape: {panel.shape}")


Found 633 files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/acc_components_extracted
Combined shape: (10354, 14)


In [2]:

# -------------------------
# 2. Clean and sort
# -------------------------
panel["Year"] = pd.to_numeric(panel["Year"], errors="coerce")
panel = panel.sort_values(["Ticker", "Year"]).reset_index(drop=True)

# Numeric columns
num_cols = ["ACT", "CHE", "LCT", "STD", "TXP", "PPEGT", "AT", "OANCF"]
for c in num_cols:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

# Fill missing STD / TXP with 0 if you want that assumption
# Comment out if you prefer to leave them missing
panel["STD"] = panel["STD"].fillna(0.0)
panel["TXP"] = panel["TXP"].fillna(0.0)


In [3]:

# -------------------------
# 3. Construct DD variables
# -------------------------
g = panel.groupby("Ticker", group_keys=False)

# Lagged total assets
panel["AT_lag1"] = g["AT"].shift(1)

# Current working capital accruals:
# WCA = (ΔCA - ΔCash) - (ΔCL - ΔSTD - ΔTP)
panel["dACT"] = g["ACT"].diff()
panel["dCHE"] = g["CHE"].diff()
panel["dLCT"] = g["LCT"].diff()
panel["dSTD"] = g["STD"].diff()
panel["dTXP"] = g["TXP"].diff()

panel["WCA"] = (panel["dACT"] - panel["dCHE"]) - (panel["dLCT"] - panel["dSTD"] - panel["dTXP"])

# CFO variables
panel["CFO_t-1"] = g["OANCF"].shift(1)
panel["CFO_t"]   = panel["OANCF"]
panel["CFO_t+1"] = g["OANCF"].shift(-1)

# Revenue change is not available in your example files.
# So this version uses only the variables you actually have.
# If you later add revenue, you can include dREV/A_{t-1} too.

# Scale by lagged total assets
scale_vars = {
    "WCA_scaled": "WCA",
    "CFO_t-1_scaled": "CFO_t-1",
    "CFO_t_scaled": "CFO_t",
    "CFO_t+1_scaled": "CFO_t+1",
    "PPE_scaled": "PPEGT",
}

for new_col, raw_col in scale_vars.items():
    panel[new_col] = panel[raw_col] / panel["AT_lag1"]

# Keep usable sample
reg_cols = [
    "Ticker", "Year",
    "WCA_scaled",
    "CFO_t-1_scaled",
    "CFO_t_scaled",
    "CFO_t+1_scaled",
    "PPE_scaled"
]

reg_df = panel[reg_cols].copy()

# Remove infinities
reg_df = reg_df.replace([np.inf, -np.inf], np.nan)

# Drop rows with missing regression inputs
reg_df = reg_df.dropna().reset_index(drop=True)

print(f"Regression sample shape: {reg_df.shape}")
print(reg_df.head())


Regression sample shape: (8702, 7)
     Ticker  Year  WCA_scaled  CFO_t-1_scaled  CFO_t_scaled  CFO_t+1_scaled  \
0  20202.OL  2019   -0.023178       -0.006868      0.075527        0.345335   
1  20202.OL  2020   -0.003304        0.018055      0.082554        0.288777   
2  20202.OL  2021    0.001364        0.060443      0.211432        0.119728   
3  20202.OL  2022    0.023256        0.208628      0.118140        0.126376   
4  20202.OL  2023   -0.024775        0.109224      0.116838        0.129075   

   PPE_scaled  
0    2.789746  
1    1.285043  
2    0.935603  
3    0.993918  
4    0.919067  


In [4]:

# -------------------------
# 4. Full OLS model
# -------------------------
y = reg_df["WCA_scaled"]
X = reg_df[["CFO_t-1_scaled", "CFO_t_scaled", "CFO_t+1_scaled", "PPE_scaled"]]
X = sm.add_constant(X)

full_model = sm.OLS(y, X).fit(cov_type="HC3")  # robust SE
print(full_model.summary())


                            OLS Regression Results                            
Dep. Variable:             WCA_scaled   R-squared:                       0.967
Model:                            OLS   Adj. R-squared:                  0.967
Method:                 Least Squares   F-statistic:                    0.9413
Date:                Mon, 13 Apr 2026   Prob (F-statistic):              0.439
Time:                        17:38:19   Log-Likelihood:                -16997.
No. Observations:                8702   AIC:                         3.400e+04
Df Residuals:                    8697   BIC:                         3.404e+04
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0737      0.123      0.

In [5]:

# -------------------------
# 5. Standardized beta model
# -------------------------
std_df = reg_df.copy()

y_std = (std_df["WCA_scaled"] - std_df["WCA_scaled"].mean()) / std_df["WCA_scaled"].std(ddof=0)

X_std = std_df[["CFO_t-1_scaled", "CFO_t_scaled", "CFO_t+1_scaled", "PPE_scaled"]].copy()
for c in X_std.columns:
    X_std[c] = (X_std[c] - X_std[c].mean()) / X_std[c].std(ddof=0)

X_std = sm.add_constant(X_std)
std_model = sm.OLS(y_std, X_std).fit(cov_type="HC3")


In [6]:

# -------------------------
# 6. Variable importance table
# -------------------------
importance_rows = []

full_r2 = full_model.rsquared

base_vars = ["CFO_t-1_scaled", "CFO_t_scaled", "CFO_t+1_scaled", "PPE_scaled"]

for var in base_vars:
    reduced_vars = [v for v in base_vars if v != var]
    
    X_red = sm.add_constant(reg_df[reduced_vars])
    reduced_model = sm.OLS(y, X_red).fit(cov_type="HC3")
    
    # For nested F-test we need non-robust versions
    full_model_nonrobust = sm.OLS(y, X).fit()
    reduced_model_nonrobust = sm.OLS(y, X_red).fit()
    
    anova_res = anova_lm(reduced_model_nonrobust, full_model_nonrobust)
    f_stat = anova_res["F"].iloc[1]
    f_pval = anova_res["Pr(>F)"].iloc[1]
    
    importance_rows.append({
        "variable": var,
        "coef": full_model.params[var],
        "robust_t": full_model.tvalues[var],
        "robust_p": full_model.pvalues[var],
        "std_beta": std_model.params[var],
        "drop_in_R2": full_r2 - reduced_model.rsquared,
        "nested_F": f_stat,
        "nested_F_p": f_pval,
    })

importance_df = pd.DataFrame(importance_rows)

# Add absolute standardized beta ranking
importance_df["abs_std_beta"] = importance_df["std_beta"].abs()

# Sort by drop in R2 first, then abs standardized beta
importance_df = importance_df.sort_values(
    ["drop_in_R2", "abs_std_beta"],
    ascending=False
).reset_index(drop=True)

print("\nVariable importance summary:")
display(importance_df)



Variable importance summary:


,variable,coef,robust_t,robust_p,std_beta,drop_in_R2,nested_F,nested_F_p,abs_std_beta
0,PPE_scaled,-0.298178,-0.767651,0.442694,-0.447725,0.024052,6393.386591,0.000000e+00,0.447725
1,CFO_t_scaled,-1.127235,-0.506066,0.612810,-1.229888,0.008905,2366.982360,0.000000e+00,1.229888
2,CFO_t-1_scaled,2.510508,1.696928,0.089710,0.193316,0.004555,1210.673427,1.793221e-248,0.193316
3,CFO_t+1_scaled,-0.191000,-0.158985,0.873681,-0.326076,0.000580,154.243196,4.063762e-35,0.326076


In [7]:

# -------------------------
# 7. Direct comparison: with vs without CFO_t+1
# -------------------------
X_no_lead = sm.add_constant(reg_df[["CFO_t-1_scaled", "CFO_t_scaled", "PPE_scaled"]])
model_no_lead = sm.OLS(y, X_no_lead).fit(cov_type="HC3")

print("\n==============================")
print("Model comparison: Full vs No CFO_t+1")
print("==============================")
print(f"Full model R^2:      {full_model.rsquared:.6f}")
print(f"No CFO_t+1 R^2:      {model_no_lead.rsquared:.6f}")
print(f"Incremental R^2:     {full_model.rsquared - model_no_lead.rsquared:.6f}")
print(f"Full model Adj R^2:  {full_model.rsquared_adj:.6f}")
print(f"No CFO_t+1 Adj R^2:  {model_no_lead.rsquared_adj:.6f}")

# Nested F-test for CFO_t+1
full_model_nonrobust = sm.OLS(y, X).fit()
model_no_lead_nonrobust = sm.OLS(y, X_no_lead).fit()
anova_cmp = anova_lm(model_no_lead_nonrobust, full_model_nonrobust)

print("\nNested F-test for adding CFO_t+1:")
display(anova_cmp)



Model comparison: Full vs No CFO_t+1
Full model R^2:      0.967282
No CFO_t+1 R^2:      0.966702
Incremental R^2:     0.000580
Full model Adj R^2:  0.967267
No CFO_t+1 Adj R^2:  0.966690

Nested F-test for adding CFO_t+1:


,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,8698.0,25780.603559,0.0,NaN,NaN,NaN
1,8697.0,25331.346590,1.0,449.256969,154.243196,4.063762e-35


In [8]:

# -------------------------
# 8. Correlation matrix of regressors
# -------------------------
corr = reg_df[["CFO_t-1_scaled", "CFO_t_scaled", "CFO_t+1_scaled", "PPE_scaled"]].corr()
print("\nRegressor correlation matrix:")
display(corr)



Regressor correlation matrix:


,CFO_t-1_scaled,CFO_t_scaled,CFO_t+1_scaled,PPE_scaled
CFO_t-1_scaled,1.000000,0.933147,0.922170,-0.817422
CFO_t_scaled,0.933147,1.000000,0.995943,-0.907739
CFO_t+1_scaled,0.922170,0.995943,1.000000,-0.925171
PPE_scaled,-0.817422,-0.907739,-0.925171,1.000000


In [9]:

# -------------------------
# 9. Optional: year-by-year importance of CFO_t+1
# -------------------------
year_results = []

for yr, sub in reg_df.groupby("Year"):
    if len(sub) < 20:
        continue
    
    y_sub = sub["WCA_scaled"]
    X_sub_full = sm.add_constant(sub[["CFO_t-1_scaled", "CFO_t_scaled", "CFO_t+1_scaled", "PPE_scaled"]])
    X_sub_red  = sm.add_constant(sub[["CFO_t-1_scaled", "CFO_t_scaled", "PPE_scaled"]])
    
    try:
        m_full = sm.OLS(y_sub, X_sub_full).fit(cov_type="HC3")
        m_red  = sm.OLS(y_sub, X_sub_red).fit(cov_type="HC3")
        
        year_results.append({
            "Year": yr,
            "n_obs": len(sub),
            "coef_CFO_t+1": m_full.params.get("CFO_t+1_scaled", np.nan),
            "t_CFO_t+1": m_full.tvalues.get("CFO_t+1_scaled", np.nan),
            "p_CFO_t+1": m_full.pvalues.get("CFO_t+1_scaled", np.nan),
            "R2_full": m_full.rsquared,
            "R2_no_lead": m_red.rsquared,
            "incremental_R2_CFO_t+1": m_full.rsquared - m_red.rsquared,
        })
    except Exception:
        continue

year_importance_df = pd.DataFrame(year_results).sort_values("Year").reset_index(drop=True)

print("\nYear-by-year importance of CFO_t+1:")
display(year_importance_df)



Year-by-year importance of CFO_t+1:


,Year,n_obs,coef_CFO_t+1,t_CFO_t+1,p_CFO_t+1,R2_full,R2_no_lead,incremental_R2_CFO_t+1
0,1999,23,0.238317,0.766773,0.443216,0.607770,0.545848,6.192227e-02
1,2000,25,-0.118417,-0.290786,0.771215,0.021567,0.010393,1.117336e-02
2,2001,30,-0.411126,-1.512457,0.130418,0.244705,0.143317,1.013882e-01
3,2002,33,0.060322,0.208075,0.835170,0.369350,0.349580,1.976961e-02
4,2003,35,-0.220827,-0.824183,0.409836,0.363994,0.058165,3.058285e-01
5,2004,53,0.893009,1.447729,0.147693,0.506004,0.346128,1.598755e-01
6,2005,250,-0.050320,-0.351428,0.725267,0.017547,0.016197,1.349943e-03
7,2006,291,0.003231,0.017442,0.986084,0.059173,0.059151,2.171294e-05
8,2007,308,1.269163,0.749397,0.453618,0.998803,0.997089,1.713901e-03
9,2008,324,0.282865,2.164367,0.030436,0.092633,0.019214,7.341894e-02


In [10]:

# -------------------------
# 10. Optional: save outputs
# -------------------------
OUT_DIR = BASE_DIR / "dd_feature_importance_output"
OUT_DIR.mkdir(exist_ok=True)

importance_df.to_csv(OUT_DIR / "variable_importance_summary.csv", index=False)
year_importance_df.to_csv(OUT_DIR / "cfo_tplus1_year_by_year_importance.csv", index=False)
reg_df.to_csv(OUT_DIR / "dd_regression_sample.csv", index=False)

print(f"\nSaved outputs to: {OUT_DIR}")


Saved outputs to: /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/dd_feature_importance_output
